# Credit RAG Model

This model is very similar to one of the models you looked at in the tasks.
A lot of times in practice, you may need specific numbers from documents such as age/income and also info that is not numerical. For this case LLMs and RAG could be a very useful tools to use, especially because they can extract unstructured information and return it in a structured way. For this example this will be transforming it into a JSON file that we can directly use to extract data from.

### Structure:

**1. Data Generation:** Similar to the tasks we will first create synthetic data on which we will train a classification model on.

**2. Classifcation Model:** The classifcaiton Model itself can be one you used in the tasks (Logistic Regression, Deep NN etc.) or something completely different.

**3. RAG pipeline:** The pretrained Model will then be passed to the RAG pipeline. Here the goal is to take an unstructured text as an input extract the key metrics used in the pretrained model, pass these too the model and then receive the outcome and explanation as the output.



# 1. Setup:

## 1.1 Installations and Imports

In [ ]:
# Installations
!pip install langchain faiss-cpu sentence-transformers langchain-community --quiet
!pip install -U langchain-google-genai --quiet
!pip install shap --quiet
!pip install unstructured --quiet
!pip install unstructured_inference --quiet
!pip install unstructured-pytesseract --quiet
!pip install --upgrade unstructured[local-inference] --quiet
!pip install poppler-utils --quiet
!apt-get install -y poppler-utils


In [ ]:
#Imports
import numpy as np
from sklearn.model_selection import train_test_split
from IPython.display import Markdown

from sklearn.linear_model import LogisticRegression
import shap

from langchain_community.document_loaders import UnstructuredPDFLoader

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
import ast

import re

In [ ]:
from google.colab import userdata
import os

os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

## 1.2 Creating Sample Data

Similar to the tasks we are going to define a probability function that defines how likely a user is to default on a loan.
For this we will use the following formula:

$ p(x) = α_0 + α_1*𝕀_{x_1<25,x_1>75}(x_1) +α_2 * x_2 +α_3*x_3 + α_4 * 𝕀_{x_4>0} + α_5 * x_4 *𝕀_{x_4>2}$,

where $x_1$ is the age, $x_2$ is the monthly income,$x_3$ is the self-employed vs salary level,  $x_4$ the number of previous defaults.

In [ ]:
m = 20000
n = 10000
n_samples = m+n

In [ ]:
x1_age = np.random.uniform(18, 80, size=n_samples)              # x1: Age
x2_income = np.random.uniform(1, 15, size=n_samples)            # x2: Monthly income (in CHF 1 000s)
x3_employment = np.random.binomial(1, 0.1, size=n_samples)      # x3: Self-employed (10% chance)
x4_defaults = np.random.poisson(0.1, size=n_samples)            # x4: Number of previous defaults
x5_sum_defaults = x4_defaults * np.random.uniform(0.3, 10.0, size=n_samples) #x5: Sum of Defaults in 1000 CHF

X = np.stack((x1_age, x2_income, x3_employment,x4_defaults,x5_sum_defaults), axis=1)


In [ ]:
sigmoid = lambda x: 1. / (1. + np.exp(-x))
alpha_0, alpha_1, alpha_2, alpha_3, alpha_4, alpha_5 = -7.5, 10, -1.1, 1., 1.75, 3.5

p_x = alpha_0 + alpha_1*(x1_age < 25) + alpha_1*(x1_age > 75) + alpha_2*x2_income + alpha_3*x3_employment + alpha_4*(x4_defaults > 0) + alpha_5*x5_sum_defaults


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(p_x, bins=20, density=True)      # density=True will normalize to a PDF
plt.xlabel('Predicted probability p(x)')
plt.ylabel('Density')
plt.title('Distribution of Predicted Probabilities')
plt.show()

In [ ]:
p_x = sigmoid(p_x)

In [ ]:
y_default = np.random.binomial(1, p_x, size=n_samples)

X_train, X_test, y_train, y_test= train_test_split(X, y_default, test_size=n, random_state=42)

##  1.3 Credit Models

For this you can use a Neural Network or basic Logistic Regression to make your decisions, these are very similar to the ones you found in the tasks previously.

### Logistic Regression:

In [ ]:
model_logres = LogisticRegression()
model_logres.fit(X_train,y_train)

logres_explainer = shap.LinearExplainer(model_logres,X_train)

### Neural Network:

This uses PyTorch. In case you may not be familiar with it, it is very well documented here: [PyTorch Documentation](https://docs.pytorch.org/docs/stable/index.html)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class NeuralNet(nn.Module):
    """
    Feedforward neural network for binary classification with sklearn-like API.

    Parameters
    ----------
    input_dim : int
        Number of input features.
    hidden_dim : int, default=20
        Neurons per hidden layer.
    num_hidden_layers : int, default=2
        Number of hidden layers.
    lr : float, default=0.001
        Learning rate for optimizer.
    batch_size : int, default=32
        Batch size for training.
    epochs : int, default=100
        Number of training epochs.
    device : str or torch.device, optional
        Device to use ('cpu' or 'cuda'). If None, auto-selects.
    verbose : bool, default=False
        If True, prints loss each epoch.
    """
    def __init__(self, input_dim, hidden_dim=20, num_hidden_layers=2,
                 lr=0.001, batch_size=32, epochs=100,
                 device=None, verbose=False):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_hidden_layers = num_hidden_layers
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.device = torch.device(device) if device is not None else torch.device(
            'cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.verbose = verbose
        # Build network
        layers = []
        in_dim = input_dim
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        layers.append(nn.Sigmoid())
        self.network = nn.Sequential(*layers).to(self.device)

    def forward(self, x):
        """
        Forward pass to compute positive-class probability.

        x : torch.Tensor, shape (n_samples, input_dim)
            Input feature batch.
        returns: torch.Tensor, shape (n_samples, 1)
        """
        if isinstance(x,np.ndarray):
          x = np.asarray(X, dtype=np.float32)
          x= torch.from_numpy(x).to(self.device)
        return self.network(x)

    def fit(self, X, y):
        """
        Train the model on data X, y.

        X : array-like, shape (n_samples, input_dim)
        y : array-like, shape (n_samples,)
        returns: self
        """
        # Prepare data
        X_np = np.asarray(X, dtype=np.float32)
        y_np = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        dataset = TensorDataset(torch.from_numpy(X_np), torch.from_numpy(y_np))
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        # Loss and optimizer
        loss_fn = nn.BCELoss()
        optimizer = torch.optim.Adam(self.network.parameters(), lr=self.lr)

        self.network.train()
        for epoch in range(1, self.epochs + 1):
            total_loss = 0
            for batch_X, batch_y in loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)
                preds = self.network(batch_X)
                loss = loss_fn(preds, batch_y)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * batch_X.size(0)
            avg_loss = total_loss / len(dataset)
            if self.verbose:
                print(f"Epoch {epoch}/{self.epochs}, Loss: {avg_loss:.4f}")
        return self

    def predict_proba(self, X):
        """
        Predict class probabilities.

        X : array-like, shape (n_samples, input_dim)
        returns: ndarray, shape (n_samples, 2)
            Columns: [P(class=0), P(class=1)].
        """
        X_np = np.asarray(X, dtype=np.float32)
        self.network.eval()
        with torch.no_grad():
            inputs = torch.from_numpy(X_np).to(self.device)
            pos = self.network(inputs).cpu().numpy().flatten()
        neg = 1.0 - pos
        return np.vstack([neg, pos]).T

    def predict(self, X, threshold=0.5):
        """
        Predict binary class labels.

        threshold : float, default=0.5
            Cutoff for positive class.
        """
        proba = self.predict_proba(X)[:,1]
        return (proba >= threshold).astype(int)


In [ ]:
model_nn = NeuralNet(input_dim = 5)
model_nn.fit(X_train,y_train)

X_train_nn = torch.from_numpy(np.asarray(X_train, dtype=np.float32))

nn_explainer = shap.DeepExplainer(model_nn,X_train_nn)

# 2. RAG Pipeline :

### 2.1 File/Report Upload:

Upload your credit report here. It should be in the a PDF document.


In [ ]:
from google.colab import files
uploaded = files.upload()
document = list(uploaded.keys())[0]

### 2.2 Pipeline Definition

The following Class is defines the main part of the pipeline. It takes in a document text of any sorts that has been extract from a PDF or similar file. Then it extracts all of the features that the Credit Model needs in a json format so it is easily processable into a numerical ```dict```. This ```dict``` is then passed to the Credit Model which returns the probability of default.

In [ ]:
class CreditRiskRAG:
    def __init__(
        self,
        model_pretrained,
        extraction_template: PromptTemplate,
        llm_model: str = "gemini-2.0-flash"
    ):
        """
        Initialize a CreditRiskRAG instance.

        Parameters
        ----------
        model_pretrained : sklearn-like estimator
            Our pretrained Credit Risk Model that we defined previously.
        extraction_template : PromptTemplate, optional
            A Promptemplate to guide the LLM to extract numeric features from the input document.
        llm_model : str, default="gemini-2.0-flash"
            The Gemini Model you want to use.
        """
        self.llm = ChatGoogleGenerativeAI(model=llm_model)
        self.extraction_template = extraction_template
        self.evaluation_model = model_pretrained

    def query(self, question: str = "", document_text: str = "") -> np.ndarray:
        """
        Extract features from `document_text` via the LLM and compute credit risk probability.

        Parameters
        ----------
        question : str, optional
            (Reserved for future use) The question to be answered.
        document_text : str
            The raw text of the credit document (e.g., financial statement, application form) as per a document loader

        Returns
        -------
        np.ndarray
            The class-probability vector from the pretrained risk model.
        """
        response_dict = self._retrieve_doc(document_text)
        probability = self.model_response(response_dict)
        return probability

    def _retrieve_doc(self, document_text: str = "") -> dict:
        """
        Use the LLM to extract structured numeric features from raw document text.

        Parameters
        ----------
        document_text : str
            Unstructured text from which to extract feature values.

        Returns
        -------
        dict
            A mapping from feature names to numeric values, parsed from the LLM’s JSON output.
        """
        prompt = self.extraction_template.format(document_text=document_text)
        response = self.llm.invoke(prompt)
        # Strip markdown-style ```json fences if present
        cleaned_response = re.sub(r'^```json\s*\n?|```$', '', response.content)

        # Safely evaluate JSON-like dict literal
        response_dict = ast.literal_eval(cleaned_response)
        return response_dict

    def model_response(self, response_dict: dict) -> np.ndarray:
        """
        Convert extracted features into a probability score using the Credit Model.

        Parameters
        ----------
        response_dict : dict
            Feature name → numeric value mapping as extracted by `_retrieve_doc`.

        Returns
        -------
        np.ndarray
            The result of `model_pretrained.predict_proba` on the feature vector (shape (1, n_classes)).
        """
        # Assemble feature vector in the order of dict values
        response_list = [response_dict['age'],response_dict['monthly_income'],response_dict['salaried'],response_dict['default_count'],response_dict['default_total']]
        X_vector = np.array(response_list)
        probability = self.evaluation_model.predict_proba(X_vector.reshape(1, -1))
        return probability


    def load_document(self,document_path:str):
        """
        Loads Document from path.

        Parameters:
        -----------
        document_path : str
          Document Path

        Returns:
        --------
        str
          Document Text
        """

        loader = UnstructuredPDFLoader(document_path)
        documents = loader.load()
        return documents[0].page_content


### 2.3 Prompt Template

For this to work properly we need a well defined prompt template for extraction. What is important here is that the prompt contains almost all exact details of how the output should be structured:

1. Ideally, you can return a sample ```dict``` with the keys already predetermined so that the LLM just has to fill in the gaps. This will almost garuantee that the output will be consitently this format without issues. On larger documents, especially when you cannot predetermine the variables you need, issues may arise with the exact JSON formatting.

2. The second big part, is that ideally you can predefine the value ranges that each variable should have and how they should be processed. This becomes especially important when some values are not explicitly found in the text but need to be implicitly extracted. Here also make sure to give the LLM an alternative if a value was not found in the data.

In [ ]:
extraction_template = PromptTemplate(
    input_variables=["context"],
    template="""
You are an information-extraction assistant.

Given the following document:
========== BEGIN DOCUMENT TEXT ============
{document_text}
========== END DOCUMENT TEXT ============

Extract the following information from the document:
- Age
- Monthly income
- Salaried status (1 if fully salaried, 0 if self-employed)
- Number of past defaults
- Total amount of past defaults

Return it as valid JSON in this exact structure:
{{'age': <age>, 'monthly_income': <monthly_income>, 'salaried': <salaried>, 'default_count': <default_count>, 'default_total': <default_total>}}

All numbers should be plain numerals.In case a value isn't specifically mentioned but can be approximated, that approximation is sufficient. If any value isn’t present, use null.

Begin extraction now:
"""
)

Now that the core components have been defined we are ready to start the process.


## 2.4 Implementation

In [ ]:
credit_model = CreditRiskRAG(model_pretrained=model_nn,
                             extraction_template=extraction_template,
                             llm_model="gemini-2.0-flash")

In [ ]:
file = credit_model.load_document(document_path = document)

In [ ]:
credit_model.query(document_text=file)

# 3. Extension: Explainability

As you saw in recent lectures, explainability and understanding of model outputs is very important to the ML workflow. In the light of those topics, there is an Addon/Extension to the previous Pipeline.
Additionally to the pipeline we showed before, we are passing:

- ```question_template```: An additional template that allows the user to ask questions about the output and ask for explanations
- ```explain_model```: the SHAP explainer model as defined inin the Credit Models section, this can be used to generate the shapely values for a given input.

Additionally in the pipeline, we also add a second part to the model response where result,shapely values, and original values get passed to an LLM and evaluated to give final response.

## 3.1 File/Report Upload:

In [ ]:
from google.colab import files
uploaded_extended = files.upload()
document_extended = list(uploaded_extended.keys())[0]

## 3.2 Pipeline Definition

In [ ]:
class CreditRiskRAGextended:
    def __init__(
        self,
        model_pretrained,
        shap_explainer,
        question_template: PromptTemplate = None,
        extraction_template: PromptTemplate = None,
        llm_model: str = "gemini-2.0-flash"
    ):
        """
        Initialize a CreditRiskRAG pipeline that extracts features from unstructured text,
        computes credit‐risk probabilities, generates SHAP explanations, and allows for
        follow-up Q&A via an LLM.

        Parameters
        ----------
        model_pretrained : object
            A pretrained credit‐risk estimator implementing `predict_proba(X)` returning
            class probabilities for each input feature vector.
        shap_explainer : callable
            A SHAP‐style explainer function or object with call signature
            `explainer(X)` that returns feature attributions for the given input vector.
        question_template : PromptTemplate, optional
            A PromptTemplate guiding the final LLM prompt to answer user questions
            conditioned on the extracted features, model probability, and explanation.
        extraction_template : PromptTemplate, optional
            A PromptTemplate guiding the LLM to extract numeric feature values
            from raw document text.
        llm_model : str, default="gemini-2.0-flash"
            The identifier of the generative LLM to use for both extraction and Q&A.
        """
        self.llm = ChatGoogleGenerativeAI(model=llm_model)
        self.extraction_template = extraction_template
        self.question_template = question_template
        self.evaluation_model = model_pretrained
        self.explain_model = shap_explainer

    def query(self, question: str = "", document_text: str = "") -> str:
        """
        Orchestrate the full pipeline: extract features, compute risk + explanation,
        and generate a user-facing answer to the provided question.

        Parameters
        ----------
        question : str, optional
            The user’s follow-up question about the credit risk or model explanation.
        document_text : str
            Raw unstructured text of the credit document (e.g., financial statements,
            application forms).

        Returns
        -------
        str
            The LLM’s answer content, formatted according to `question_template`, incorporating
            the extracted feature values, predicted risk probability, and SHAP explanation.
        """
        response_dict = self._retrieve_doc(document_text)
        probability, explanation = self.model_response(response_dict)

        prompt = self.question_template.format(
            question=question,
            information_dict=response_dict,
            probability=probability,
            explanation=explanation
        )
        answer = self.llm.invoke(prompt)
        return answer.content

    def _retrieve_doc(self, document_text: str = "") -> dict:
        """
        Use the LLM to parse raw text into a structured dict of numeric features.

        Parameters
        ----------
        document_text : str
            Unstructured credit document content from which to extract feature values.

        Returns
        -------
        dict
            Mapping from feature names (str) to their extracted numeric values (int/float),
            parsed from the LLM’s JSON-style output.
        """
        prompt = self.extraction_template.format(document_text=document_text)
        response = self.llm.invoke(prompt)

        # Remove markdown json fences if present
        cleaned = re.sub(r'^```json\s*\n?|```$', '', response.content)
        feature_dict = ast.literal_eval(cleaned)
        return feature_dict

    def model_response(self, response_dict: dict) -> tuple:
        """
        Convert the extracted feature dict into a probability vector and SHAP explanation.

        Parameters
        ----------
        response_dict : dict
            Feature name → numeric value mapping as produced by `_retrieve_doc`.

        Returns
        -------
        tuple
            - probability : np.ndarray
              The output of `model_pretrained.predict_proba`, shape (1, n_classes).
            - explanation : Any
              The SHAP explanation object returned by calling `shap_explainer` on the feature vector.
        """
        response_list = [response_dict['age'],response_dict['monthly_income'],response_dict['salaried'],response_dict['default_count'],response_dict['default_total']]
        X_vector = np.array(response_list)
        probability = self.evaluation_model.predict_proba(X_vector.reshape(1, -1))

        X_vector = X_vector.reshape(1, -1)

        if isinstance(self.explain_model, shap.DeepExplainer):
            # X_vector needs to be converted to torch if the torch nn is used.
            X_vector = torch.from_numpy(np.asarray(X_vector,dtype=np.float32))

        explanation = self.explain_model(X_vector)
        return probability, explanation

    def load_document(self,document_path:str):
        loader = UnstructuredPDFLoader(document_path)
        documents = loader.load()
        return documents[0].page_content

## 3.3 Prompt Templates:

1. ```extraction_template```: Document extraction instructions. The same as from the previous section
2. ```question_template```: This template takes your question, the information from the text, the model output probability and the shapely values and returns the answer to the question. This can be used for tasks such as explaining why a certain profile got the probability that it did.

In [ ]:
extraction_template = PromptTemplate(
    input_variables=["context"],
    template="""
            You are an information-extraction assistant.

            Given the following document:
            ========== BEGIN DOCUMENT TEXT ============
            {document_text}
            ========== END DOCUMENT TEXT ============

            Extract the following information from the document:
            - Age
            - Monthly income
            - Salaried status (1 if fully salaried, 0 if self-employed)
            - Number of past defaults
            - Total amount of past defaults

            Return it as valid JSON in this exact structure:
            {{'age': <age>, 'monthly_income': <monthly_income>, 'salaried': <salaried>, 'default_count': <default_count>, 'default_total': <default_total>}}

            All numbers should be plain numerals.In case a value isn't specifically mentioned but can be approximated, that approximation is sufficient. If any value isn’t present, use null.

            Begin extraction now:
            """
            )

In [ ]:
question_template = PromptTemplate(input_variables=["question","information_dict","probability","explanation"],
                                 template= """
                                 You are an information extraction assistant.
                                 Given the information:
                                 {information_dict}
                                 the pretrained machine learning model predicted the following proabbility of default: {probability}.
                                 Additionaly here are the shapely values to explain why the model came to the conclusion that it did.
                                 {explanation}

                                 Please respond in a natural way. Explain it as if you are explaining to a non-technical audience.
                                 Explicity mentions of the shapely values are not necessary, they are simply there to help you understand the model output.

                                 Answer the following question:
                                 {question}

                                 Give a thorough response similar to a report. Return the output in markdown please.
                                 """
)

## 3.4 Implementation:

In [ ]:
credit_model = CreditRiskRAGextended(model_pretrained=model_nn,shap_explainer = nn_explainer ,extraction_template = extraction_template, question_template=question_template)


In [ ]:
file_extended = credit_model.load_document(document_path = document_extended)

In [ ]:
output = credit_model.query(question = "Why did this applicant get the loan decision that it got?",document_text = file_extended)


In [ ]:
display(Markdown(output))